# Analyse des déterminants de l'équipement en Véhicules Électriques

## 1. Introduction

Ce projet vise à comprendre quels facteurs (socio-économiques, infrastructures existantes) influencent le taux d'équipement en véhicules électriques au niveau communal en France.

Nous fusionnons trois sources de données provenant de data.gouv.fr :
- IRVE : Données sur les infrastructures de recharge.
- INSEE : Revenus et données socio-économiques.
- Immatriculations : Parc de VE par commune.

## 2. Chargement et Exploration des Données Brutes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loading import load_all_datasets, charger_communes
from src.cleaning import standardize_all_codes, clean_irve_variables_finales, corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal, garder_derniere_observation_commune
from src.features import preparer_base_ve, creer_features_irve, creer_taux_equipement_ve
from src.modelisation import modelisation_lasso, modelisation_xgboost
from src.utils import diagnostic_cle_jointure, creer_gdf_irve, joindre_communes, ajouter_codes_geo, compter_valeurs_manquantes, compter_uniques

# Configuration des chemins
PATHS = {
    'irve': 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435',
    'revenu': 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',
    've': 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
}

# Chargement
df_irve_raw, df_revenu_raw, df_ve_raw = load_all_datasets(PATHS)

#### IRVE

In [ ]:
print(f"Shape : {df_irve_raw.shape}")
df_irve_raw.sample(5)

Shape : (217227, 52)


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
119632,Power Dot France,891118473.0,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,0176310684,Power Dot France,FRPD1PACTNCB,NaN,Action - Neufchâtel-en-Bray,...,1bf98bac-94a9-4909-8726-47a203038a40,power-dot-france,2023-01-30T13:44:34.220000+00:00,1.435405,49.736249,76270.0,Neufchâtel-en-Bray,True,True,True
149574,R3 SPV COM,949415814.0,avanoost@r3-charge.fr,R3,avanoost@r3-charge.fr,tel:+33-1-23-45-67-89,R3 SPV COM,FRR3MP1063497,FR*R3M*1063497,Grande-Synthe - Marie Blachère,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,2.310551,51.023085,59760.0,Grande-Synthe,True,True,False
48461,MISSING,NaN,NaN,Enerstock | FR*ENT,l.gailland@enerstock.fr,NaN,Enerstock,FRENTP97297494394931708,1406038,Enerstock/679a02924f5da094d714fd99,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,2.400505,43.497440,NaN,NaN,False,False,False
142034,JANEC EVS,883799462.0,contact@janec.fr,QOVOLTIS,contact@qovoltis.com,09 74 34 03 40,QOVOLTIS,FRQOVP5600004,FR*QOV*56-00004,17 RUE DE BERDER _ HOTELLE PARC FETAN 56870 L,...,f16b5634-f3d1-4c48-9168-db17ce8d506c,qovoltis,2024-12-23T15:52:31.707000+00:00,-2.895187,47.585394,56870.0,Larmor-Baden,True,True,False
133457,GROUPE INDIGO,NaN,NaN,GROUPE INDIGO,support.emobility.fr@group-indigo.com,NaN,Réseau de recharge du groupe Indigo,FRPKGP13202001,NaN,Les Terrasses du Port,...,d445f73a-797f-46ea-98cc-2927ec7a018d,indigo-group,2025-08-13T13:39:08.570000+00:00,5.365791,43.307302,NaN,NaN,False,False,False


Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 212234 lignes pour 52 variables.

#### Véhicules

In [ ]:
print(f"Shape : {df_ve_raw.shape}")
df_ve_raw.sample(5)

Shape : (703545, 8)


,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP
444953,11340,SAINTE-EULALIE,200035715,CA Carcassonne Agglo,2022-09-30,2,0,502
557574,10370,SOLIGNY-LES-ÉTANGS,200006716,CC du Nogentais,2021-03-31,0,0,299
392997,59301,HERGNIES,245901160,CA Valenciennes Métropole,2023-09-30,84,1,5339
354913,26376,VILLEPERDRIX,200068229,CC des Baronnies en Drôme Provençale,2023-12-31,3,0,138
46154,39052,BIEF-DES-MAISONS,200069623,CC Champagnole Nozeroy Jura,2021-03-31,0,0,134


Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.
Cette table comporte 703545 lignes pour 8 variables. 

#### Revenus

In [ ]:
print(f"Shape : {df_revenu_raw.shape}")
df_revenu_raw.sample(5)

Shape : (34926, 57)


,Nom géographique GMS,Code géographique,Libellé géographique,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Nbre d'unités de consommation dans les ménages fiscaux,[DISP] 1ᵉʳ quartile (€),[DISP] Médiane (€),[DISP] 3ᵉ quartile (€),[DISP] Écart interquartile (€),...,[DEC] 9ᵉ décile (€),[DEC] Rapport interdécile D9/D1,[DEC] S80/S20,[DEC] Iice de Gini,[DEC] Part des revenus d’activité (%),[DEC] dont part des salaires et traitements (%),[DEC] dont part des iemnités de chômage (%),[DEC] dont part des revenus des activités non salariées (%),"[DEC] Part des pensions, retraites et rentes (%)",[DEC] Part des autres revenus (%)
177,Lapeyrouse,01207,Lapeyrouse,131.0,352.0,227.6,NaN,27830.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2857,Bédeilhac-et-Aynat,09045,Bédeilhac-et-Aynat,90.0,173.0,128.7,NaN,20750.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3277,Épothémont,10139,Épothémont,77.0,166.0,116.6,NaN,22940.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3401,Les Noës-près-Troyes,10265,Les Noës-près-Troyes,1478.0,3038.0,2159.5,14000.0,18750.0,23860.0,9870.0,...,32430.0,"6,2",8.0,0.345,60.0,52.4,4.0,3.6,37.6,2.4
17531,La Tieule,48191,La Tieule,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

## 3. Nettoyage et Standardisation Géographique

Le défi majeur de ce projet est la fusion des données. 

#### Choix de la variable de jointure
Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus
La jointure sera faite sur ces variables.

#### Préparation des variables de jointure
Cependant les codes INSEE des communes sont souvent mal formatés.

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

#### Unification des codes
Nous utilisons la fonction standardize_all_codes pour garantir que chaque base dispose d'une colonne code_geo homogène sur 5 caractères.

In [ ]:
dfs_to_clean = {
    "irve": (df_irve_raw, "code_insee_commune"),
    "revenu": (df_revenu_raw, "Code géographique"),
    "ve": (df_ve_raw, "CODGEO")
}

cleaned = standardize_all_codes(dfs_to_clean)
df_irve = cleaned["irve"]
df_revenu = cleaned["revenu"]
df_ve = cleaned["ve"]

Standardisation du code INSEE pour : irve (colonne : code_insee_commune)
Standardisation du code INSEE pour : revenu (colonne : Code géographique)
Standardisation du code INSEE pour : ve (colonne : CODGEO)


Une partie des données IRVE ne possède pas de code INSEE commune, mais dispose de coordonnées latitude/longitude. Plutôt que de supprimer ces lignes, nous utilisons un référentiel géographique pour retrouver la commune correspondante.
Grâce au croisement spatial avec les données de Cartiflette, nous avons pu attribuer un code commune à la quasi-totalité des points de charge.

In [ ]:
# 1. Téléchargement des contours de communes via Cartiflette
communes_fr = charger_communes()

# 2. Transformation de l'IRVE en données géographiques (GeoDataFrame)
gdf_irve = creer_gdf_irve(df_irve, long_col="consolidated_longitude", lat_col="consolidated_latitude")

# 3. Jointure spatiale pour identifier la commune par les coordonnées GPS
gdf_result = joindre_communes(gdf_irve, communes_fr)

# 4. Intégration du code récupéré (code_geo_total) dans le dataframe principal
df_irve = ajouter_codes_geo(df_irve, gdf_result)

There was an error while reading the file from the URL: https://minio.lab.sspcloud.fr/projet-cartiflette/production/provider=IGN/dataset_family=ADMINEXPRESS/source=EXPRESS-COG-CARTO-TERRITOIRE/year=2022/administrative_level=COMMUNE/crs=4326/DEPARTEMENT=20/vectorfile_format=geojson/territory=metropole/simplification=0/raw.geojson
Error message: '/vsimem/pyogrio_c4455c95ecbe44daa769915370bddf84' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.


Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

Valeurs manquantes :
{'code_insee_commune': np.int64(65645), 'code_geo_total': np.int64(1857)}

Valeurs uniques :
{'code_insee_commune': 9536, 'code_geo_total': 10317}


Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux. En effet, les données entrées n'étant pas toujours vérifiées, des erreurs étaient présentes. Certaines code géographiques correspondaient au code postal par exemple.

## 4. Feature Engineering : Passage à l'échelle communale

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [ ]:
list(df_irve.columns)

['nom_amenageur',
 'siren_amenageur',
 'contact_amenageur',
 'nom_operateur',
 'contact_operateur',
 'telephone_operateur',
 'nom_enseigne',
 'id_station_itinerance',
 'id_station_local',
 'nom_station',
 'implantation_station',
 'adresse_station',
 'code_insee_commune',
 'coordonneesXY',
 'nbre_pdc',
 'id_pdc_itinerance',
 'id_pdc_local',
 'puissance_nominale',
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'gratuit',
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',
 'condition_acces',
 'reservation',
 'horaires',
 'accessibilite_pmr',
 'restriction_gabarit',
 'station_deux_roues',
 'raccordement',
 'num_pdl',
 'date_mise_en_service',
 'observations',
 'date_maj',
 'cable_t2_attache',
 'last_modified',
 'datagouv_dataset_id',
 'datagouv_resource_id',
 'datagouv_organization_or_owner',
 'created_at',
 'consolidated_longitude',
 'consolidated_latitude',
 'consolidated_code_postal',
 'consolidated_commune',
 '

In [ ]:
list(df_ve.columns)

['CODGEO',
 'LIBGEO',
 'EPCI',
 'LIBEPCI',
 'DATE_ARRETE',
 'NB_VP_RECHARGEABLES_EL',
 'NB_VP_RECHARGEABLES_GAZ',
 'NB_VP',
 'code_geo']

In [ ]:
list(df_revenu.columns)

['Nom géographique GMS',
 'Code géographique',
 'Libellé géographique',
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 "[DISP] Nbre d'unités de consommation dans les ménages fiscaux",
 '[DISP] 1ᵉʳ quartile (€)',
 '[DISP] Médiane (€)',
 '[DISP] 3ᵉ quartile (€)',
 '[DISP] Écart interquartile (€)',
 '[DISP] 1ᵉʳ décile (€)',
 '[DISP] 2ᵉ décile (€)',
 '[DISP]3ᵉ décile (€)',
 '[DISP] 4ᵉ décile (€)',
 '[DISP] 6ᵉ décile (€)',
 '[DISP] 7ᵉ décile (€)',
 '[DISP] 8ᵉ décile (€)',
 '[DISP] 9ᵉ décile (€)',
 '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
 '[DISP] S80/20',
 '[DISP] Iice de Gini',
 '[DISP] Part des revenus d’activité (%)',
 '[DISP] dont part des salaires et traitements(%)',
 '[DISP] dont part des iemnités de chômage (%)',
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des pensions, retraites et rentes (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)',
 "[DISP] Part de l'ensemble des prestat

Nos bases n'ont pas la même unité d'observation. L'IRVE liste des bornes (plusieurs par ville), alors que nous voulons une analyse par commune.

Pour pouvoir efectuer la jointure sur les codes géographiques, il est essentiel que pour chacune de nos 3 bases de données, chaque ligne corresponde à un unique code géographique.

Il faut alors réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

Il faut également étudier la base df_ve afin de supprimer les "doublons" de code géographique.

#### IRVE
Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger. L'objectif final est d'expliquer ou prédire le taux de véhicules électriques local.

Après étude des nos variables (cf etude_df_irve.ipynb) nous avons sélectionner plusieurs variables nous permettant d'en créer de nouvelles, agrégées par code géographique.

Certaines variables ont été écarté directement car jugées non pertinente pour notre sujet, d'autres présetaient trop de valeurs manquantes, d'autres étaient trop peu informatives ou encore était du texte de libre donc trop difficile à utiliser avec le peu de temps dont nous disposons.

------------------------ début -------------

variables utilisées (12 variables) pour la création des variables agrégées :`code_geo_total`,`nom_operateur`,
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'gratuit',
               'paiement_cb',
               'paiement_autre'

variables créées : une ligne finale = un code géographique.

|variables utilisées| Variable finale   | Construction      |
|| ----------------- | ----------------- |
|| total_pdc         | somme             |
|| puissance_moyenne | moyenne           |
|| puissance_max     | max               |
|| nb_operateurs     | nunique           |
|| top_operateur     | mode              |
|| pct_type_2        | moyenne booléenne |
|| pct_gratuit       | moyenne booléenne |
|| part_voirie       | dummies + moyenne |

------------------------ fin -------------

Commençons par rendre les données des variables sélectionnées propres avant de faire l'agrégation.

In [ ]:
# Traitement IRVE : Transformation des types et agrégation
df_irve_clean = clean_irve_variables_finales(df_irve)
df_irve_final = creer_features_irve(df_irve_clean)
df_irve_final.sample(5)

NameError: name 'pd' is not defined

#### Véhicules
La base véhicules contient plusieurs dates d'observation pour une même commune.

Afin d'être cohérent avec les autres sources de données (IRVE, INSEE), nous souhaitons disposer d'une seule ligne par commune correspondant à la situation la plus récente disponible.

Nous conservons donc, pour chaque code commune (`CODGEO`), la dernière observation temporelle.

In [ ]:
# Traitement VE : Sélection de la dernière observation
df_ve_final = preparer_base_ve(df_ve)
df_ve_final.head()

(vérifier ça)
------------------------ début -------------

Deux variables sont conservées :

- `NB_VP` : nombre total de voitures particulières
- `NB_VP_RECHARGEABLES_EL` : nombre de voitures rechargeables / électriques

Plutôt que d'utiliser le nombre brut de véhicules électriques, nous construisons un ratio :

\[
taux\_equipement\_ve = \frac{NB\_VP\_RECHARGEABLES\_EL}{NB\_VP}
\]

Ce ratio est plus pertinent car il neutralise l'effet de taille des communes.

In [ ]:
df_ve_latest["taux_equipement_ve"].describe().round(4)

------------------------ fin -------------

## 5. Jointure

------------------------ début -------------

##### df_revenus

Voici notre première sélection de variables :

In [ ]:
var_selec = ['Code géographique',
'[DISP] Médiane (€)',
 '[DISP] Iice de Gini',
 #garder uniquement l'un des deux
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 #zone dynamique économiquement (potentiellement plus jeune aussi)
 '[DISP] Part des revenus d’activité (%)',
 #plus riche
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)'
]

df_revenus_final=df_revenus[var_selec]

**Variables à garder :**
[DISP] Médiane (€), [DISP] Part des revenus d’activité (%), [DISP] Iice de Gini.

Intérêt : La médiane du revenu est indispensable pour vérifier si l'adoption du VE est corrélée à la richesse de la commune. L'indice de Gini peut montrer si les communes très inégalitaires ont un comportement différent.

...

------------------------ fin -------------

In [ ]:
# Fusion des trois bases
df_final = (
    df_ve_final
    .merge(df_irve_final, left_on="CODGEO", right_on="code_geo_total", how="left")
    .merge(df_revenus_final, left_on="CODGEO", right_on="Code géographique", how="left")
)

print("Shape base finale :", df_final.shape)
df_final.sample(5)

## 6. Préparation de la base de données

------------------------ début -------------

In [ ]:
# Nombre de valeurs manquantes par variable
df_final.isna().sum()

Cette nouvelle base de données comporte 35197 communes.
Parmis ces communes 24919 ne contiennet pas de données provenant de df_irve.
Cela représente environ 70% de nos communes. Après étude de la réalité, nous considéront que ces 24919 sont sans point de charge. En effet ces chiffres sont crédible car une grande partie des communes françaises ont :
- peu d’habitants
- peu de commerces
- peu de parkings publics
- peu de flux routiers
donc l'absence de borne est très fréquente.

Les IRVE sont souvent concentrées dans :
villes
axes routiers
zones commerciales
parkings publics
intercommunalités centrales
Donc beaucoup de petites communes voisines n’en ont aucune.

Attention :
Une petite commune peut ne pas avoir de borne mais être à 5 km d’une commune équipée.
Donc absence locale ne signifie pas absence d’accès réel.

In [ ]:
df_imputation = df_final.copy()

# Liste des colonnes numériques de df_irve (comptage, puissance, pourcentages)
cols_to_zero = [
    'total_pdc', 
    'puissance_moyenne', 
    'puissance_max', 
    'nb_operateurs', 
    'pct_type_2', 
    'pct_combo_ccs', 
    'pct_chademo', 
    'pct_type_ef', 
    'pct_gratuit', 
    'pct_paiement_cb', 
    'pct_paiement_autre',
    'Parking priv\x8e r\x8eserv\x8e \x88 la client\x8fle', 
    'Parking priv\x8e \x88 usage public', 
    'Parking privé réservé à la clientèle', 
    'Parking privé à usage public', 
    'Parking public', 
    'Station dédiée à la recharge rapide', 
    'Voirie'
]

# 1. Remplacement par 0 pour les chiffres
df_imputation[cols_to_zero] = df_imputation[cols_to_zero].fillna(0)

# 2. Remplacement par un label pour le texte
df_imputation['top_operateur'] = df_imputation['top_operateur'].fillna('Aucun')

df_imputation.sample(5)

In [ ]:
missing = pd.DataFrame({
    "nb_manquants": df_imputation.isna().sum(),
    "pct_manquants": (df_imputation.isna().mean() * 100).round(2)
})

missing.sort_values("nb_manquants", ascending=False)

[DISP] Iice de Gini
[DISP] dont part des revenus des activités non salariées (%)
[DISP] Part des revenus du patrimoine et autres revenus (%)
[DISP] Part des revenus d’activité (%)

Il va falloir enlever ces variables... trop de valeurs manquantes !

------------------------ fin -------------

## 7. Analyse des variables

Après le choix des variables : 
0. variable cible = taux de VE
1. faire des analyses descriptives : 
- automatiser avec une fonction
- matrice de corrélation (heatmap)
2. modélisation :
simple regression ou plus avace (random forest...)

## 8. Modélisation

Nous cherchons à prédire la variable taux_equipement_ve.

Approche 1 : Régression Lasso
Le modèle Lasso est idéal pour l'interprétabilité. Il permet de voir quelles variables "pèsent" réellement dans la décision d'achat d'un véhicule électrique.

Approche 2 : XGBoost
Ce modèle non-linéaire permet ...

Nos variables pour la modélisation sont les suivantes :

attention LASSO ne prend pas de valeurs manquantes !!

In [ ]:
features = ['total_pdc', 'puissance_max', 'nb_operateurs', 'pct_gratuit']
target = 'taux_equipement_ve' 

df_model = df_final[features + [target]].dropna()

# 2. Exécuter le Lasso (Baseline & Sélection)
model_l, metrics_l, coeffs_l = modelisation_lasso(df_model, features, target)
print(f"R2 Lasso : {metrics_l['r2']:.3f}")
print("Déterminants (Lasso) :\n", coeffs_l[coeffs_l != 0])

# 3. Exécuter le Gradient Boosting (Performance)
model_x, metrics_x, importance_x = modelisation_xgboost(df_final, features, target)
print(f"R2 XGBoost : {metrics_x['r2']:.3f}")
importance_x.plot(kind='barh', title="Importance des variables - XGBoost")

In [ ]:
features = ['[DISP] Médiane (€)', '[DISP] Iice de Gini', 'total_pdc', 'puissance_moyenne']
target = 'taux_equipement_ve'

model_lasso, metrics_lasso, coeffs = modelisation_lasso(df_master, features, target)
model_xgb, metrics_xgb, importance = modelisation_xgboost(df_master, features, target)

## 9. Analyse des résultats